# Idealista Web Scraper — Valencia Real Estate

## Purpose
Scrape property listings from the public Idealista website to supplement the API collector.
The API imposes monthly request quotas; the web scraper provides an unlimited parallel data source.

## Usage
1. Install dependencies: `pip install cloudscraper beautifulsoup4 lxml`
2. Run cells sequentially — no AWS credentials required for local execution.
3. Output is saved to `data/s3/` as JSON files matching the API collector envelope format.

## Target URLs
```
Rent:  https://www.idealista.com/alquiler-viviendas/valencia-valencia/con-de-50-metros-cuadrados,hasta-200-metros-cuadrados,con-ascensor,buen-estado/
Sale:  https://www.idealista.com/venta-viviendas/valencia-valencia/con-de-50-metros-cuadrados,hasta-200-metros-cuadrados,con-ascensor,buen-estado/
Page N: ?pagina=N
```

## Search Parameters
- **Location:** Valencia city center
- **Property type:** Homes, 50–200 m², elevator, good preservation
- **Operations:** `rent` (alquiler) and `sale` (venta)

In [1]:
"""Imports for Idealista web scraper."""

import json
import logging
import random
import re
import time
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

import cloudscraper
from bs4 import BeautifulSoup

# Configure logging for notebook
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [2]:
class IdealistaScraperError(Exception):
    """Custom exception for Idealista web scraper errors.

    Raised for HTTP failures, parse failures, and rate-limiting events.
    """

    pass

In [ ]:
class ScraperConfig:
    """Configuration for Idealista web scraper search parameters.

    Mirrors the SearchConfig pattern from the API collector.
    Holds base URLs, search filters, and output settings.

    Attributes:
        BASE_URL: Root domain for Idealista Spain.
        OPERATION_MAP: Maps operation names to Spanish URL segments.
        FILTERS: URL path segment encoding search filters.
        output_dir: Local directory for JSON output.
        max_pages: Maximum pages to scrape per operation (None = all).
        request_delay_range: Tuple of (min, max) seconds between requests.
    """

    BASE_URL: str = "https://www.idealista.com"

    # Maps operation to Spanish URL path segment
    OPERATION_MAP: Dict[str, str] = {
        "rent": "alquiler-viviendas",
        "sale": "venta-viviendas",
    }

    LOCATION: str = "valencia-valencia"

    # URL-encoded search filters — wider range than API since no quota limits:
    # 50-200 m², elevator, good preservation
    FILTERS: str = "con-de-50-metros-cuadrados,hasta-200-metros-cuadrados,con-ascensor,buen-estado"

    def __init__(
        self,
        output_dir: str = "../../data/s3",
        max_pages: Optional[int] = None,
        request_delay_range: tuple = (2.0, 4.5),
    ) -> None:
        """Initialize scraper configuration.

        Args:
            output_dir: Local directory to write JSON output files.
            max_pages: Maximum number of pages to scrape per operation.
                None means scrape all available pages.
            request_delay_range: Tuple of (min_seconds, max_seconds)
                for random delay between page requests.
        """
        self.output_dir = output_dir
        self.max_pages = max_pages
        self.request_delay_range = request_delay_range

    def build_url(self, operation: str, page: int = 1) -> str:
        """Build the search URL for a given operation and page number.

        Args:
            operation: Either 'sale' or 'rent'.
            page: Page number (1-indexed).

        Returns:
            Full URL string for the search results page.

        Raises:
            IdealistaScraperError: If operation is not 'sale' or 'rent'.
        """
        if operation not in self.OPERATION_MAP:
            raise IdealistaScraperError(
                f"Invalid operation '{operation}'. Must be 'sale' or 'rent'."
            )

        op_segment = self.OPERATION_MAP[operation]
        url = f"{self.BASE_URL}/{op_segment}/{self.LOCATION}/{self.FILTERS}/"

        if page > 1:
            url += f"?pagina={page}"

        return url

In [4]:
@dataclass
class ParsedListing:
    """A single property listing parsed from an Idealista search result card.

    Field names mirror the Idealista API `elementList` schema so that
    downstream analysis notebooks can consume both sources uniformly.
    Fields not available on search-result cards default to None.

    Attributes:
        property_code: Unique Idealista listing identifier.
        price: Listing price in euros.
        size: Property size in square meters.
        url: Full URL to the listing detail page.
        operation: Either 'rent' or 'sale'.
        rooms: Number of rooms (may be None from search cards).
        floor: Floor number as string (may be None).
        address: Street address text.
        neighborhood: Neighborhood name.
        thumbnail: URL of the listing thumbnail image.
        price_by_area: Price per square meter.
        has_lift: Whether the building has an elevator.
        latitude: Geographic latitude (None from search cards).
        longitude: Geographic longitude (None from search cards).
        district: District name (None from search cards).
        bathrooms: Number of bathrooms (None from search cards).
        description: Listing description (None from search cards).
    """

    property_code: str
    price: Optional[float] = None
    size: Optional[float] = None
    url: str = ""
    operation: str = ""
    rooms: Optional[int] = None
    floor: Optional[str] = None
    address: Optional[str] = None
    neighborhood: Optional[str] = None
    thumbnail: Optional[str] = None
    price_by_area: Optional[float] = None
    has_lift: Optional[bool] = None
    latitude: Optional[float] = None
    longitude: Optional[float] = None
    district: Optional[str] = None
    bathrooms: Optional[int] = None
    description: Optional[str] = None

    def to_dict(self) -> Dict[str, Any]:
        """Convert to dictionary with camelCase keys matching Idealista API schema.

        Returns:
            Dictionary with camelCase keys and typed values.
            Missing fields are explicitly set to None (not omitted).
        """
        return {
            "propertyCode": self.property_code,
            "price": self.price,
            "size": self.size,
            "url": self.url,
            "operation": self.operation,
            "rooms": self.rooms,
            "floor": self.floor,
            "address": self.address,
            "neighborhood": self.neighborhood,
            "thumbnail": self.thumbnail,
            "priceByArea": self.price_by_area,
            "hasLift": self.has_lift,
            "latitude": self.latitude,
            "longitude": self.longitude,
            "district": self.district,
            "bathrooms": self.bathrooms,
            "description": self.description,
        }

In [5]:
# CSS selectors for Idealista search result pages.
# Centralised here so that markup changes require only updating this dict.
DOM_SELECTORS: Dict[str, str] = {
    "listing_cards": "article.item-multimedia-container",
    "link": "a.item-link",
    "price": "span.item-price",
    "detail_chars": "span.item-detail",
    "address": ".item-detail-char--truncated",
    "thumbnail": "img.item-multimedia",
    "next_page": "a.icon-arrow-right-after",
}

In [6]:
class SearchResultsParser:
    """Parse Idealista search result HTML into structured listings.

    Uses the DOM_SELECTORS mapping to locate elements. Each listing card
    is parsed into a ParsedListing dataclass instance.

    Attributes:
        selectors: CSS selector mapping used for element lookup.
    """

    def __init__(self, selectors: Optional[Dict[str, str]] = None) -> None:
        """Initialize parser with optional custom selectors.

        Args:
            selectors: CSS selector dict. Defaults to DOM_SELECTORS.
        """
        self.selectors = selectors or DOM_SELECTORS

    def parse(self, html: str, operation: str) -> List[ParsedListing]:
        """Parse search results HTML into a list of ParsedListing objects.

        Args:
            html: Raw HTML string of a search results page.
            operation: Either 'rent' or 'sale'.

        Returns:
            List of ParsedListing instances extracted from the page.
        """
        # TODO: Implement in subtask 1.2
        raise NotImplementedError("Parser implementation is part of subtask 1.2")

    def has_next_page(self, html: str) -> bool:
        """Check whether the search results page has a next-page link.

        Args:
            html: Raw HTML string of a search results page.

        Returns:
            True if a next-page navigation element is found.
        """
        # TODO: Implement in subtask 1.2
        raise NotImplementedError("Next-page check is part of subtask 1.2")

    @staticmethod
    def _extract_property_code(url: str) -> str:
        """Extract the property code from an Idealista listing URL.

        Args:
            url: Listing URL, e.g. '/inmueble/12345678/'

        Returns:
            The numeric property code string.

        Raises:
            IdealistaScraperError: If code cannot be extracted.
        """
        match = re.search(r"/inmueble/(\d+)/", url)
        if not match:
            raise IdealistaScraperError(f"Cannot extract property code from URL: {url}")
        return match.group(1)

    @staticmethod
    def _parse_price(text: str) -> Optional[float]:
        """Parse price from text like '1.200€/mes' or '250.000€'.

        Args:
            text: Raw price text from the page.

        Returns:
            Price as float, or None if parsing fails.
        """
        try:
            cleaned = re.sub(r"[^\d]", "", text.split("€")[0].replace(".", ""))
            return float(cleaned) if cleaned else None
        except (ValueError, IndexError):
            return None

    @staticmethod
    def _parse_size(text: str) -> Optional[float]:
        """Parse size in m² from detail text like '115 m²'.

        Args:
            text: Raw detail text from the page.

        Returns:
            Size as float, or None if parsing fails.
        """
        match = re.search(r"(\d+)\s*m", text)
        return float(match.group(1)) if match else None

    @staticmethod
    def _parse_rooms(text: str) -> Optional[int]:
        """Parse room count from detail text like '3 hab.'.

        Args:
            text: Raw detail text from the page.

        Returns:
            Room count as int, or None if parsing fails.
        """
        match = re.search(r"(\d+)\s*hab", text)
        return int(match.group(1)) if match else None

    @staticmethod
    def _parse_floor(text: str) -> Optional[str]:
        """Parse floor from detail text like 'Planta 4ª'.

        Args:
            text: Raw detail text from the page.

        Returns:
            Floor as string, or None if parsing fails.
        """
        match = re.search(r"[Pp]lanta\s*(\w+)", text)
        if match:
            return match.group(1).replace("ª", "")
        match = re.search(r"(\d+)ª", text)
        return match.group(1) if match else None

In [7]:
def save_locally(
    listings: List[ParsedListing],
    operation: str,
    page: int,
    total_pages: int,
    output_dir: str,
) -> Path:
    """Save scraped listings to a local JSON file.

    Writes a JSON envelope matching the existing API collector format so
    that downstream analysis notebooks can consume both sources uniformly.

    Args:
        listings: List of parsed listing dataclass instances.
        operation: Either 'rent' or 'sale'.
        page: Current page number.
        total_pages: Total number of pages scraped.
        output_dir: Directory path for output files.

    Returns:
        Path to the written JSON file.
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

    envelope = {
        "operation": operation,
        "source": "web_scraper",
        "collected_at": datetime.utcnow().isoformat() + "Z",
        "page": page,
        "totalPages": total_pages,
        "itemsPerPage": len(listings),
        "elementList": [listing.to_dict() for listing in listings],
    }

    filename = f"scraper_{operation}_{timestamp}_{page}.json"
    filepath = output_path / filename

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(envelope, f, ensure_ascii=False, indent=2)

    logger.info("Saved %d listings to %s", len(listings), filepath)
    return filepath

In [8]:
# --- Configuration ---
# Adjust these settings before running the scraper.

OUTPUT_DIR = "../../data/s3"  # Relative to this notebook's location
MAX_PAGES = 2  # Limit pages during development; set to None for full scrape

config = ScraperConfig(
    output_dir=OUTPUT_DIR,
    max_pages=MAX_PAGES,
    request_delay_range=(2.0, 4.5),
)

# Quick validation
print("Rent URL (page 1):", config.build_url("rent", page=1))
print("Sale URL (page 2):", config.build_url("sale", page=2))
print(f"Output directory:  {config.output_dir}")
print(f"Max pages:         {config.max_pages}")

Rent URL (page 1): https://www.idealista.com/alquiler-viviendas/valencia-valencia/con-de-100-metros-cuadrados,hasta-160-metros-cuadrados,con-ascensor,buen-estado/
Sale URL (page 2): https://www.idealista.com/venta-viviendas/valencia-valencia/con-de-100-metros-cuadrados,hasta-160-metros-cuadrados,con-ascensor,buen-estado/?pagina=2
Output directory:  ../../data/s3
Max pages:         2
